In [ ]:
# name_extraction_spacy.ipynb

In [ ]:
pip install -U spacy-transformers

In [ ]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

In [ ]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.7.2/en_core_web_trf-3.7.2-py3-none-any.whl

In [1]:
import re
from collections import Counter
from pathlib import Path

import spacy
import pandas as pd

In [2]:
# Load spaCy NER model
nlp = spacy.load("en_core_web_trf") 

In [3]:
paths = {
    "The Count of Monte Cristo": "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/The Count of Monte Cristo.txt",
    "In Search of the Castaways": "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/In search of the castaways.txt",
}

TITLE_STOP = {
    "Mr", "Mrs", "Miss", "M", "Mme", "Mlle",
    "Doctor", "Dr", "Sir", "Lord", "Lady",
    "Count", "Captain", "Baron", "Viscount",
    "Madame", "Monsieur",
}

GLOBAL_EXCLUDE = {
    "Chapter", "Book", "Illustration", "Sidenote",
    "United States", "United States of America",
}

In [4]:
def normalize_person_name(text: str) -> str:
    """
    Normalize spaCy PERSON span:
    - drop leading titles (Lord, Count, etc.)
    - strip possessive 's
    - strip trailing punctuation
    """
    tokens = text.strip().split()
    if not tokens:
        return ""

    # Drop leading titles
    while tokens and tokens[0].rstrip(".,'") in TITLE_STOP:
        tokens = tokens[1:]
    if not tokens:
        return ""

    # Strip possessive 's
    tokens = [re.sub(r"'s$", "", t) for t in tokens]

    name = " ".join(tokens)
    name = re.sub(r"[.,;:!?]+$", "", name).strip()

    if not name:
        return ""
    if name in GLOBAL_EXCLUDE:
        return ""

    return name

In [5]:
def merge_name_variants(counter: Counter) -> Counter:
    """
    Merge variants like:
      'Edmond Dantès', 'Dantès', 'Edmond'
    into a combined count, biased towards the longest form.
    """
    merged = Counter()

    # First, keep original counts
    for name, cnt in counter.items():
        merged[name] += cnt

    # Build index by last token
    last_to_full = {}
    for name, cnt in counter.items():
        parts = name.split()
        if len(parts) > 1:
            last = parts[-1]
            last_to_full.setdefault(last, Counter())
            last_to_full[last][name] += cnt

    # For each last name, pick the most frequent full form
    alias_map = {}
    for last, fulls in last_to_full.items():
        # most common full name for that last token
        canonical, _ = fulls.most_common(1)[0]
        alias_map[last] = canonical

    # Now, rebuild merged counts
    final = Counter()
    for name, cnt in merged.items():
        parts = name.split()
        if len(parts) == 1 and name in alias_map:
            # map bare 'Dantès' -> 'Edmond Dantès'
            final[alias_map[name]] += cnt
        else:
            final[name] += cnt

    return final

In [6]:
def extract_person_counts_with_spacy(book_path: str, batch_chars: int = 200000,
                                     min_count: int = 3) -> pd.DataFrame:
    """
    Use spaCy NER + nlp.pipe to extract PERSON entities and count names.
    """
    text = Path(book_path).read_text(encoding="utf-8", errors="ignore")
    name_counter = Counter()

    # Prepare chunks for nlp.pipe
    chunks = [text[i:i+batch_chars] for i in range(0, len(text), batch_chars)]

    for doc in nlp.pipe(chunks, batch_size=4):  # adjust batch_size as needed
        for ent in doc.ents:
            if ent.label_ != "PERSON":
                continue
            name = normalize_person_name(ent.text)
            if not name:
                continue
            name_counter[name] += 1

    # Merge variants like 'Dantès' + 'Edmond Dantès'
    merged = merge_name_variants(name_counter)

    items = [(name, cnt) for name, cnt in merged.items() if cnt >= min_count]
    df = pd.DataFrame(sorted(items, key=lambda x: -x[1]), columns=["name", "count"])
    return df

In [7]:
for book, fname in paths.items():
    print("Processing:", book)
    df = extract_person_counts_with_spacy(fname, batch_chars=200000, min_count=3)
    out_name = f"{book.replace(' ','_').replace('\"','')}_name_counts.csv"
    df.to_csv(out_name, index=False)
    print("Saved:", out_name, "with", len(df), "names")

Processing: The Count of Monte Cristo


/usr/local/lib/python3.12/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


Saved: The_Count_of_Monte_Cristo_name_counts.csv with 194 names
Processing: In Search of the Castaways
Saved: In_Search_of_the_Castaways_name_counts.csv with 50 names
